# Solution: Probabilistic PCA

In [ ]:
import os
from urllib.request import urlretrieve
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from cycler import cycler
import seaborn as sns
import torch
from torch.distributions.multivariate_normal import MultivariateNormal
from torch.distributions.wishart import Wishart

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["image.cmap"] = LinearSegmentedColormap.from_list(
    "color_map", [colors[10], "#FFFFFF", colors[9]]
)
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

torch.set_default_tensor_type(torch.DoubleTensor)

## Principal component analysis (PCA)

The classic PCA can be interpreted as an orthogonal projection of the data onto a lower dimensional linear space, known as the *principal subspace*. To find the subspace that captures the maximum variance of the data, we have to find the eigenvectors (or PCA-modes) of the data-covariance matrix. The coordinates of a data point in the original space is given by a linear combination of the PCA-modes and the PCA-coeffecients.

In [ ]:
torch.manual_seed(0)

# set params for data
D = 2  # dimension of variable x
M = 2  # dimension of latent variable z

# generate data
C = Wishart(2, torch.eye(2)).sample()
C = C / torch.linalg.matrix_norm(C)
x = MultivariateNormal(torch.zeros(2), C).sample(torch.Size([200]))

# get eigenvalues and eigenvectors
eig_val, eig_vec = torch.linalg.eigh(C)

# plot eigenvectors and eigenvalues alongside data
fig, ax = plt.subplots(figsize=(6, 4))
origin = np.array([[0.0], [0.0]])
ax.scatter(*x.T[0:2], s=10)
[
    ax.quiver(
        0,
        0,
        np.sqrt(eig_val[i]) * eig_vec[i, 0],
        np.sqrt(eig_val[i]) * eig_vec[i, 1],
        scale_units="xy",
        scale=0.3,
    )
    for i in range(2)
]

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.axis("equal")
ax.set_ylim(-3.5, 3.5)
plt.show()

## Probabilistic PCA

The problem of finding the principal subspace can also be tackled by introducing a probabilistic latent variable model. The classic PCA from before then arises as the maximum likelihood solution to this problem. The probabilistic reformulation, however, has some advantages (take a look at Bishop 12.2.) over the deterministic approach. Your task is the compute the statistics of a lower dimensional embedding of the oberserved variable $\boldsymbol{x}$.

The following quantities are needed to determine the marginal distribution of the data $p(\boldsymbol{x})$:
- Projection matrix $\boldsymbol{W}_{ML}$
- Data mean $\bar{\boldsymbol{x}}$
- Variance of the observation noise $\sigma_{ML}^2$

With those quantities, we can write out $p(\boldsymbol{x}) \sim \mathcal{N}(\boldsymbol{x}|\bar{\boldsymbol{x}}, \boldsymbol{WW}^T + \sigma^2 \boldsymbol{I})$

In [ ]:
def ppca(x, M):
    """
    Probabilistic PCA
    :param x: (Tensor) The observation
    :param M: (int) latent variable dimension
    :return: Tuple:
                (Tensor) Projection matrix
                (Tensor) data mean
                (float) variance of observation noise
                (Tensor) covariance matrix
    """

    # ---------------------- student exercise --------------------------------- #
    D = x.shape[1]
    # get covariance
    cov = torch.cov(x.T)

    # get eigenvectors and eigenvalues
    eig_val, eig_vec = torch.linalg.eigh(cov)
    order = torch.flip(torch.argsort(eig_val), dims=[0])
    eig_val, eig_vec = eig_val[order], eig_vec[:, order]

    # compute quantities needed to get principal components
    U_m = eig_vec[:, :M]
    L_m = torch.diag(eig_val[:M])
    sig2_ml = torch.sum(eig_val[M:]) / (D - M)

    # compute components, mean, and covariance of ppca
    W_ml = torch.matmul(U_m, torch.sqrt(L_m - sig2_ml * np.eye(M)))
    mu_ml = torch.mean(x, axis=0)
    C_ml = torch.matmul(W_ml, W_ml.T) + sig2_ml * torch.eye(D)
    # ---------------------- student exercise --------------------------------- #

    return W_ml, mu_ml, sig2_ml, C_ml

Once we obtained our maximum likelihood estimates for $\boldsymbol{W}$ and $\sigma$, we can project the data onto the subspace. For the PPCA, that means to evaluate the posterior distribution for the latent variable $p(\boldsymbol z | \boldsymbol x )$. Compare the posterior mean to the projection obtained from the deterministic PCA. More specifically, try to answer the following questions:

- What happens to the posterior mean for the PPCA model in the limit $\sigma^2 \rightarrow 0$
- What effect does an increasing $\sigma$ have on the posterior mean?

### Arbitrary multivariate Normal distribution

Let us try out the model on data from multivariate normal distribution with full covariance structure. In this simple 2D case, PPCA should be able to exaclty recover the data distribution. We can use this property to test our implementation.

In [ ]:
torch.manual_seed(0)

D = 2
M = 1
N = 500

# get samples from some D-dimensional gaussian with full covariance
C = Wishart(D, torch.eye(D)).sample()
C = C / torch.linalg.matrix_norm(C)
x = MultivariateNormal(torch.zeros(D), C).sample(torch.Size([N]))

# compute ppca
W_ml, mu_ml, sig2_ml, C_ml = ppca(x, M)

# define grid to plot 2D marginal of ppca model
n_grid = 500
C_max = np.sqrt(np.max(np.diag(C)))
lims = [-3.0 * C_max, 3 * C_max]
grid = torch.vstack(
    [
        tensor.flatten()
        for tensor in torch.meshgrid(
            *[torch.linspace(*lims, n_grid) for i in range(2)], indexing="xy"
        )
    ]
)

# get samples of ppca model
test_dims = [0, 1]
p_grid = torch.exp(
    MultivariateNormal(mu_ml[test_dims], C_ml[test_dims, :][:, test_dims]).log_prob(
        grid.T
    )
)
levels = torch.linspace(torch.min(p_grid), torch.max(p_grid), 10)[1:]

# get figure and plot pdf
fig, ax = plt.subplots(figsize=(6, 4))
cmap = sns.color_palette("Blues", as_cmap=True)
markersize = 7

ax.contourf(
    *grid.reshape(-1, n_grid, n_grid),
    p_grid.reshape(n_grid, n_grid),
    cmap=cmap,
    levels=levels,
)

# generate and plot samples from ppca model
x_ppca = MultivariateNormal(mu_ml, C_ml).sample(torch.Size([N]))

ax.scatter(
    *x_ppca.T[test_dims], color="C4", label="$x_{ppca}$", s=markersize, alpha=0.5
)

# plot original samples
ax.scatter(*x.T[test_dims], color="C5", label="$x_{true}$", s=markersize, alpha=0.4)

ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel(rf"$x_{{{test_dims[0] + 1}}}$")
ax.set_ylabel(rf"$x_{{{test_dims[1] + 1}}}$")
ax.legend()

plt.show()

### PPCA with the expectation-maximization algorithm

Instead of directly computing the eigenvectors of the data covariance matrix, the PPCA parameters can also be obtained with the expectation-maximization (EM) algorithm that you already know from the Gaussian mixtures notebook.
It estimates the joint distribution of the latent and the observed variables in the expectation step. In the maximization step, it then optimizes for the parameters — in our case the projection matrix $\boldsymbol{W}$ and the variance of the observation noise $\sigma^2$.

While the EM algorithm is not necessary to obtain the posterior, it can be helpful for performing PPCA on large datasets.
It bypasses the computation of the full data covariance matrix, which can become very costly for large and/or high-dimensional datasets.
Further, it is a necessary step towards the more powerful non-linear latent variable models in the following notebook.

Recall from the lecture that we introduce an approximate distribution $q_{\phi}(\boldsymbol{z}|\boldsymbol{x}) \approx p_{\theta}(\boldsymbol{z}|\boldsymbol{x})$.
Our proposal $q_{\phi}(\boldsymbol{z} | \boldsymbol{x})$ depends on some inital guess of the model parameters $\boldsymbol{\phi} = \left[\boldsymbol{W}_{old}^T, \sigma^2_{old} \right]^T$.
We compute the expected mean and covariance of the posterior $q_{\phi}(\boldsymbol{z}|\boldsymbol{x})$ using those old values.
This is the E-step.
We then iteratively update these parameters until the Kullback-Leibler divergence is sufficiently small, which in turn means that we maximized the evidence lower bound (ELBO), which at the end of the optimization matches the model evidence.
This is the M-step.
Your task is to complete the implementation of the EM algorithm.
Take a look at the lecture slides for details on the E- and M-steps.

In [ ]:
def ppca_em(x, M, maxiter=20):
    """
    Probabilistic PCA with EM
    :param x: (Tensor) The observation
    :param M: (int) latent variable dimension
    :return: Tuple:
                (Tensor) Projection matrix
                (Tensor) data mean
                (float) variance of observation noise
    """

    N = x.shape[0]
    D = x.shape[1]

    # initial values
    W = torch.rand(D, M)
    sig2 = torch.rand(1)
    mu = torch.mean(x, axis=0)

    # posterior moments
    expec_z = torch.zeros(N, M)
    expec_zzt = torch.zeros(N, M, M)

    # ---------------------- student exercise --------------------------------- #
    for iter in range(maxiter):
        # Likelihood
        C = W @ W.T + sig2 * torch.eye(D)
        Cinv = torch.linalg.inv(C)
        error = 0.0
        for i in range(N):
            error += torch.inner(x[i, :] - mu, Cinv @ (x[i, :] - mu))
        elbo = (
            -float(N) * float(D) / 2.0 * torch.log(torch.tensor(2.0 * torch.pi))
            - float(N) / 2.0 * torch.log(torch.linalg.det(C))
            - error / 2.0
        )

        # E-step
        Minv = torch.linalg.inv(W.T @ W + sig2 * torch.eye(M))
        for i in range(N):
            expec_z[i, :] = torch.inner(Minv @ W.T, x[i, :] - mu)
            expec_zzt[i, :, :] = sig2 * Minv + expec_z[i, :] @ expec_z[i, :].T

        # M-step
        sum_left = torch.zeros(D, M)
        for i in range(N):
            sum_left += torch.outer(x[i, :] - mu, expec_z[i, :])

        W = sum_left @ torch.linalg.inv(torch.sum(expec_zzt, dim=0))
        sig2 = 0

        for i in range(N):
            sig2 += (
                torch.linalg.norm(x[i, :] - mu) ** 2
                - 2 * torch.inner(expec_z[i, :].T @ W.T, x[i, :] - mu)
                + torch.trace(expec_zzt[i, :, :] @ W.T @ W)
            )

        sig2 /= float(N) * float(D)

        print(f"EM iteration: {iter + 1}, ELBO: {elbo:.2f}")
    # ---------------------- student exercise --------------------------------- #

    return W, mu, sig2

In [ ]:
torch.manual_seed(0)

D = 2
M = 1
N = 500

# get samples from some D-dimensional gaussian with full covariance
C = Wishart(D, torch.eye(D)).sample()
C = C / torch.linalg.matrix_norm(C)
x = MultivariateNormal(torch.zeros(D), C).sample(torch.Size([N]))

# compute ppca
W_em, mu_em, sig2_em = ppca_em(x, M, maxiter=20)

print(f"\nProjection matrix W:\n ML: {W_ml}\n EM: {W_ml}\n diff: {W_ml - W_em}\n")
print(
    f"Observation noise sig2:\n ML: {sig2_ml:.6f}\n EM: {sig2_ml:.6f}\n diff: {sig2_ml - sig2_em}"
)

### PCA as a visualization tool

The dimensionality reduction can also be used for visualization purposes.
It might be challenging to extract patterns from individual components of the data vector $\boldsymbol{x}$.
PCA projects the dataset onto a subspace and, therefore, considers contributions from all components.
Looking at the first and second components of the latent representation $z_1$ and $z_2$, respectively, tells us how present the first and second PCA modes are for a given data point.

To illustrate the idea, we load the **MNIST** dataset, consisting of images of variations of the handwritten digits 1-9.
The images have a resolution of 28x28 pixels.
Take a look at what happens if we select two random pixels of all images of 0s and 1s and plot them, taking their values as coordinates:

In [ ]:
# load mnist images (if necessary)
url = "https://surfdrive.surf.nl/s/JSmQkKrq87EGnXp/download"
filename = "mnist_images.dat"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

mnist = torch.tensor(np.loadtxt(filename))

# load mnist labels (if necessary)
url = "https://surfdrive.surf.nl/s/CjS4fkERFBr5zmk/download"
filename = "mnist_labels.dat"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

mnist_labels = torch.tensor(np.loadtxt(filename))

# find the data points that correspond to all 0's and 1's
idx_0 = torch.where(mnist_labels == 0)
idx_1 = torch.where(mnist_labels == 1)
mnist_sub = torch.vstack((mnist[idx_0], mnist[idx_1]))

# plot the values of two pixels [180, 464] for all remaining imgages
fig, ax = plt.subplots()
pixels = [180, 464]
ax.scatter(*mnist[idx_0][:, pixels].T, c="C0", s=5, label="0")
ax.scatter(*mnist[idx_1][:, pixels].T, c="C1", s=5, label="1")
ax.legend()
plt.show()

You can see that the data points are scattered all across the domain, and many ended up on the boundary.
We even have trouble finding a good location for the legend.
There is no obvious separation between the two classes looking at these two componens.
Try to find a combination of pixels that offers some more structure to work with.

We now turn to PCA to visualize this dataset. We will use the `ppca` function you implemented earlier.
The following plot contains the first and second component of the latent representation of the same data.
It is easy to see that the two classes form two clusters in this representation.

In [ ]:
# compute ppca for dataset
W_ml, mu_ml, sig2_ml, C_ml = ppca(mnist_sub, 2)

# project data onto latent space
z = np.zeros((mnist.shape[0], 2))
for i in range(mnist.shape[0]):
    z[i] = torch.linalg.inv(W_ml.T @ W_ml) @ W_ml.T @ (mnist[i] - mu_ml)

# plot in latent space
fig, ax = plt.subplots()
ax.scatter(*z[idx_0].T, c="C0", s=5, label="0")
ax.scatter(*z[idx_1].T, c="C1", s=5, label="1")
ax.legend()
plt.show()

## PPCA on non-Gaussian data

Probabilistic PCA assumes a gaussian disribution over the observed variable, but it can also be applied to data that is not normally distributed. Let us apply the model on the **mechanical MNIST Cahn-Hilliard** data set.

In [ ]:
# Download the Cahn-Hilliard dataset (if necessary)
url = "https://surfdrive.surf.nl/s/Jis2zsxZSigqJod/download"
filename = "25x25.dat"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

# Load the dataset
cahn_hilliard = torch.tensor(np.loadtxt(filename))

nrows, ncols = 3, 3
fig, ax = plt.subplots(nrows, ncols, figsize=(2.5 * nrows, 2.5 * ncols))

for i in range(9):
    ax.flat[i].imshow(cahn_hilliard[7 * i + 21].reshape(25, 25))
    ax.flat[i].set_axis_off()

In [ ]:
torch.manual_seed(0)

M = 10
N = 10000

# get first N images of cahn-hilliard dataset
x = cahn_hilliard[:N]

# compute ppca
W_ml, mu_ml, sig2_ml, C_ml = ppca(x, M)

# get figure
K = 4
fig, ax = plt.subplots(K, 2, figsize=(5, 2.5 * K))

z = np.zeros((K, M))

# compute latent representation of image and convert it back to original space
for i in range(K):
    ax[i, 0].imshow(x[i].reshape(25, 25))
    z = torch.linalg.inv(W_ml.T @ W_ml) @ W_ml.T @ (x[i] - mu_ml)
    x_s = torch.matmul(W_ml, z)  # + rng.normal(scale=np.sqrt(sig2_ml), size=len(x[0]))
    ax[i, 1].imshow(x_s.reshape(25, 25))

[axs.set_axis_off() for axs in ax.flat]
ax[0, 0].set_title("full image")
ax[0, 1].set_title("ppca mean")

plt.show()

### Generating data in latent space

Let us now exploit the generative nature of the model to create new microstructures that are similar in distribution to the dataset. For that purpose, you should first  sample from the latent variable $\boldsymbol{z} \sim \mathcal{N}(\boldsymbol{z}|0, I)$. You obtain a sample of $\boldsymbol{x}$ by plugging $\boldsymbol{z}$ it into the definition of the observed variable $\boldsymbol{x} = \boldsymbol{W}\boldsymbol{z} + \boldsymbol{\mu} + \boldsymbol{\epsilon}$

In [ ]:
torch.manual_seed(0)

M = 10
N = 10000

# get first N images of cahn-hilliard dataset
x = cahn_hilliard[:N]

# compute ppca
W_ml, mu_ml, sig2_ml, C_ml = ppca(x, M)

# get figure
K = 4
fig, ax = plt.subplots(1, K, figsize=(2.5 * K, 2.5))

for i in range(K):
    # ---------------------- student exercise --------------------------------- #
    # sample latent representation of image and convert it back to original space
    z = torch.normal(0, 1, torch.Size([M]))
    x_s = (
        torch.matmul(W_ml, z)
        + mu_ml
        + torch.normal(0.0, np.sqrt(sig2_ml), torch.Size([len(x[0])]))
    )
    # ---------------------- student exercise --------------------------------- #

    ax[i].imshow(x_s.reshape(25, 25))

[axs.set_axis_off() for axs in ax.flat]
[axs.set_title(f"Sample {i + 1}") for i, axs in enumerate(ax.flat)]

plt.show()

Our samples from the latent space do not really resemble any of the structures in the dataset.
Try to come up with an explanation for the model's poor generative performance.
A good starting point are the assumptions we made about the structure of the dataset.
Try to support your claims with numerical evidence.


<!-- solution -->
### Solution

One reason for the weak generative performance is that our dataset is too far away from being normally distributed (see histograms below).
The datasets guassianity is, however, a key assumption of the PPCA model, which in the end is just a linear gaussian model.
In such a scenario, we can resort to more capable, non-linear generative models, such as variational autoencoders.
You can take a swing at its implementation in the next notebook.
<!-- solution -->

In [ ]:
# <!-- solution -->

n_rows, n_cols = 4, 3
n_dims = n_rows * n_cols
fig, ax = plt.subplots(
    n_rows,
    n_cols,
    figsize=(n_cols * 2.5, n_rows * 2.0),
    constrained_layout=True,
    sharey=False,
    sharex=True,
)

for i in range(n_dims):
    pixel = 5 * i + 200
    ax.flat[i].hist(cahn_hilliard[:, pixel], bins=25)
    ax.flat[i].set_title(f"pixel {pixel + 1}", fontsize=11)

# <!-- solution -->